# Reconstruccion de los archivos FASTA (`lncRNA.fa` y `miRNA.fa`)

Este notebook **regenera desde cero** los dos archivos FASTA de secuencias completas
que necesita `generate_csv.ipynb` (y `generate_csv_filtered.ipynb`).

`generate_csv.ipynb` los carga con `read_fa()` y los usa **unicamente para obtener la
longitud** de cada secuencia (`lnc_len`, `mir_len`), que son las dos primeras columnas
de la matriz de entrenamiento.

## Archivos de ENTRADA
- `data/sequences/seq+ss_join_<lncRNA>.txt`       -> linea 1 = secuencia del lncRNA
- `data/sequences_miRNA/seq+ss_join_<miRNA>.txt`  -> linea 1 = secuencia del miRNA (precursor)
- `data/txt_interac/mirnas_lncrnas_validated_positive_pairs.txt`  (opcional, para filtrar)
- `data/txt_interac/negative_pairs.txt`                           (opcional, para filtrar)

## Archivos de SALIDA
- `data/fasta/fasta_seq/lncRNA.fa`
- `data/fasta/fasta_seq/miRNA.fa`

Formato FASTA estandar (`>nombre` seguido de la secuencia), compatible con
`Bio.SeqIO.parse(..., format="fasta")`.

> **Nota sobre los miRNA:** la secuencia en disco es el *precursor* completo
> (minusculas = flancos, mayusculas = miRNA maduro). `MIRNA_REGION` controla si
> se escribe el precursor completo o solo la region madura. Debe coincidir con la
> opcion usada en `generate_kmer_dicts.ipynb` para mantener la coherencia.


In [20]:
import os

# ====================== CONFIGURACION ======================
# Rutas relativas a la carpeta src/ (igual que el resto de notebooks del proyecto)
SEQ_LNC_DIR = "./../data/sequences"        # secuencias de lncRNA (seq+ss_join_*.txt)
SEQ_MIR_DIR = "./../data/sequences_miRNA"  # secuencias de miRNA  (seq+ss_join_*.txt)

OUT_LNC_FA = "./../data/fasta/fasta_seq/lncRNA.fa"
OUT_MIR_FA = "./../data/fasta/fasta_seq/miRNA.fa"

PAIRS_FILES = [
    "./../data/txt_interac/mirnas_lncrnas_validated_positive_pairs.txt",
    "./../data/txt_interac/negative_pairs.txt",
]

# ONLY_PAIRS:
#   True  -> solo se escriben los RNA que aparecen en los pares de interaccion
#            (rapido y suficiente para generate_csv.ipynb).
#   False -> se escriben TODAS las secuencias de las carpetas (archivos grandes).
ONLY_PAIRS = True

# MIRNA_REGION: la secuencia del miRNA en disco es el PRECURSOR.
#   "full"   -> escribe el precursor completo
#   "mature" -> escribe solo la region madura (letras en mayuscula)
# Debe coincidir con lo usado en generate_kmer_dicts.ipynb.
MIRNA_REGION = "full"
# ============================================================

In [21]:
PREFIX = "seq+ss_join_"


def read_sequence(path, mature_only=False):
    """Lee la linea 1 del archivo seq+ss_join_*.txt y la normaliza a alfabeto ACGT."""
    with open(path) as f:
        seq = f.readline().strip()
    if mature_only:                        # conserva solo la region en mayusculas
        seq = "".join(c for c in seq if c.isupper())
    return seq.upper().replace("U", "T")


def write_fasta(seq_dir, out_path, keep=None, mature_only=False):
    """Recorre seq_dir y escribe un archivo FASTA estandar (>nombre + secuencia)."""
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    written, skipped = 0, 0
    found_names = set()
    with open(out_path, "w") as out:
        for fname in sorted(os.listdir(seq_dir)):
            if not fname.startswith(PREFIX):
                continue
            name = os.path.splitext(fname[len(PREFIX):])[0]
            if keep is not None and name not in keep:
                skipped += 1
                continue
            seq = read_sequence(os.path.join(seq_dir, fname), mature_only)
            out.write(f">{name}\n{seq}\n")
            written += 1
            found_names.add(name)
    print(f"  -> {out_path}")
    print(f"     {written} secuencias escritas, {skipped} archivos omitidos")
    return found_names

In [22]:
# Conjunto de RNA que realmente se usan en el entrenamiento (pares de interaccion)
needed_lnc, needed_mir = set(), set()
for pf in PAIRS_FILES:
    if os.path.exists(pf):
        for line in open(pf):
            parts = line.strip().split(",")
            if len(parts) >= 2:
                needed_lnc.add(parts[0])
                needed_mir.add(parts[1])
print(f"lncRNA en pares: {len(needed_lnc)}   |   miRNA en pares: {len(needed_mir)}")

lncRNA en pares: 3165   |   miRNA en pares: 267


In [23]:
# --- Generar lncRNA.fa ---
keep_lnc = needed_lnc if ONLY_PAIRS else None
print("Generando lncRNA.fa ...")
found_lnc = write_fasta(SEQ_LNC_DIR, OUT_LNC_FA, keep=keep_lnc)

Generando lncRNA.fa ...
  -> ./../data/fasta/fasta_seq/lncRNA.fa
     3165 secuencias escritas, 118084 archivos omitidos


In [24]:
# --- Generar miRNA.fa ---
keep_mir = needed_mir if ONLY_PAIRS else None
print("Generando miRNA.fa ...")
found_mir = write_fasta(SEQ_MIR_DIR, OUT_MIR_FA, keep=keep_mir,
                        mature_only=(MIRNA_REGION == "mature"))

Generando miRNA.fa ...
  -> ./../data/fasta/fasta_seq/miRNA.fa
     267 secuencias escritas, 2389 archivos omitidos


In [25]:
# --- Verificacion de cobertura y formato ---
missing_lnc = needed_lnc - found_lnc
missing_mir = needed_mir - found_mir

print("=== VERIFICACION DE COBERTURA ===")
print(f"lncRNA de pares sin archivo de secuencia: {len(missing_lnc)}")
print(f"miRNA  de pares sin archivo de secuencia: {len(missing_mir)}")
if missing_lnc:
    print("  lncRNA faltantes (muestra):", sorted(missing_lnc)[:10])
if missing_mir:
    print("  miRNA  faltantes (muestra):", sorted(missing_mir)[:10])

# Comprobacion del formato leyendo los .fa con la misma funcion que generate_csv.ipynb
from Bio import SeqIO

def read_fa(path):
    res = {}
    for x in SeqIO.parse(path, format="fasta"):
        res[str(x.id)] = str(x.seq).replace("U", "T")
    return res

lnc_fa = read_fa(OUT_LNC_FA)
mir_fa = read_fa(OUT_MIR_FA)
print("\n=== FORMATO DE SALIDA ===")
ej_lnc = next(iter(lnc_fa))
ej_mir = next(iter(mir_fa))
print(f"lncRNA.fa: {len(lnc_fa)} secuencias | ej: {ej_lnc} (longitud {len(lnc_fa[ej_lnc])})")
print(f"miRNA.fa : {len(mir_fa)} secuencias | ej: {ej_mir} (longitud {len(mir_fa[ej_mir])})")

if not missing_lnc and not missing_mir:
    print("\nOK: todos los RNA de los pares tienen su secuencia en el .fa.")
else:
    print("\nATENCION: hay RNA sin secuencia; los pares que los usen "
          "fallaran en generate_csv.ipynb (KeyError).")

=== VERIFICACION DE COBERTURA ===
lncRNA de pares sin archivo de secuencia: 0
miRNA  de pares sin archivo de secuencia: 0

=== FORMATO DE SALIDA ===
lncRNA.fa: 3165 secuencias | ej: NONHSAT000043.2 (longitud 2513)
miRNA.fa : 267 secuencias | ej: hsa-let-7a-5p (longitud 80)

OK: todos los RNA de los pares tienen su secuencia en el .fa.


In [26]:
%pip install biopython

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
